# Chapter 2 — Vectors and Vector Geometry: SageMath Companion

This is the symbolic, exact-arithmetic counterpart to `code/python/02_vectors_and_geometry.ipynb`. Where the Python notebook used NumPy floats and pictures, this notebook uses SageMath's exact arithmetic, native `vector`, symbolic dot products, and exact angles.

**To run this notebook:** open it in [CoCalc](https://cocalc.com/) (no install needed) — *File → Open from URL* and paste the raw GitHub URL — or run locally with the SageMath kernel (`sage -n jupyter`).

**Kernel:** SageMath.

**Layout:**

1. Vectors over QQ — exact arithmetic and norms
2. The dot product, three identities checked symbolically
3. The angle between vectors — *exact* this time
4. Projection formula — derived AND verified
5. Orthogonal decomposition with Pythagoras (exact)
6. Cauchy–Schwarz — symbolic proof in ℝ²
7. Parametric line, plotted with arrows
8. Standard unit vectors and the `e_i` family
9. **Exercises** for you to solve
10. Solutions

## 1. Vectors over QQ — exact arithmetic

Sage's `vector(QQ, ...)` gives you exact rationals. The norm `‖v‖` may involve a square root; Sage keeps it symbolic instead of rounding.

In [ ]:
u = vector(QQ, [3, 4])
v = vector(QQ, [4, 3])

show('u =', u, '   v =', v)
show('u + v =', u + v)
show('2u - v =', 2*u - v)
show('|u| =', u.norm(), '   |v| =', v.norm())
print('u lives in', u.parent())

Notice `u.norm()` returned `5` exactly, not `5.0`. This matters when chains of computations are supposed to simplify down to a clean answer.

## 2. The three properties of the dot product, checked symbolically

From the notes (§2.6.1):

1. **Symmetry:** `u · v = v · u`
2. **Linearity in each slot:** `(c u + w) · v = c (u · v) + (w · v)`
3. **Positive-definiteness:** `v · v = ‖v‖²`

Sage can check all three for *generic* vectors with symbolic entries — proving them, not just sampling.

In [ ]:
var('u1 u2 u3 v1 v2 v3 w1 w2 w3 c')
# norm() involves abs(), which Sage leaves unevaluated unless it knows the entries are real.
# Without these assumptions, |v|² stays as abs(v1)² + … and won't simplify to v1² + ….
for sym in [u1, u2, u3, v1, v2, v3, w1, w2, w3, c]:
    assume(sym, 'real')

u_sym = vector(SR, [u1, u2, u3])
v_sym = vector(SR, [v1, v2, v3])
w_sym = vector(SR, [w1, w2, w3])

# Symmetry
print('Symmetry  u·v == v·u :',
      bool(u_sym.dot_product(v_sym) == v_sym.dot_product(u_sym)))

# Linearity in the first slot
lhs = (c*u_sym + w_sym).dot_product(v_sym)
rhs = c * u_sym.dot_product(v_sym) + w_sym.dot_product(v_sym)
print('Linearity (c u + w)·v == c(u·v)+(w·v) :',
      bool((lhs - rhs).expand() == 0))

# Positive-definiteness as identity v·v == |v|^2
print('v · v == |v|^2 :',
      bool(v_sym.dot_product(v_sym) == v_sym.norm()^2))

All three pass. These are no longer empirical observations — Sage just verified the identity holds for *every* choice of components.

## 3. The angle between vectors — *exact*

Where NumPy gives `θ = 0.6435...`, Sage can give `θ = arccos(11/(5·√10))` exactly — and convert to degrees with `n(...)` only when you ask.

In [ ]:
u = vector(QQ, [1, 2])
v = vector(QQ, [3, 4])

cos_theta = u.dot_product(v) / (u.norm() * v.norm())
show('cos θ =', cos_theta)
show('θ      =', arccos(cos_theta))
show('θ° (numerical) ≈', n(arccos(cos_theta) * 180 / pi))

And special angles come out exactly. The angle between `(1,0)` and `(1,1)` is `π/4` symbolically:

In [ ]:
a = vector(QQ, [1, 0])
b = vector(QQ, [1, 1])
show('cos θ =', a.dot_product(b) / (a.norm() * b.norm()))
show('θ     =', arccos(a.dot_product(b) / (a.norm() * b.norm())))

## 4. Projection formula — derived *and* verified

From §2.8.1 the projection coefficient is `α = (u · v) / (v · v)`, derived from the perpendicularity condition `(u − α v) · v = 0`. Sage can both **solve for α** and **verify** the resulting decomposition.

In [ ]:
var('alpha')
var('u1 u2 v1 v2')
u_sym = vector(SR, [u1, u2])
v_sym = vector(SR, [v1, v2])

# Perpendicularity condition
remainder = u_sym - alpha * v_sym
perp_eq = remainder.dot_product(v_sym) == 0
show('Perpendicularity equation: ', perp_eq)

# Solve for α
alpha_sol = solve(perp_eq, alpha)[0]
show('Solution: ', alpha_sol)

# Match to the textbook formula
expected = u_sym.dot_product(v_sym) / v_sym.dot_product(v_sym)
print('Matches the formula α = (u·v)/(v·v)?',
      bool(alpha_sol.rhs() - expected == 0))

Sage *re-derived* the projection formula from a single perpendicularity equation. That's the entire logical content of §2.8.1 reduced to one `solve(...)` call.

## 5. Orthogonal decomposition with Pythagoras (exact)

Take a concrete `u, v` over QQ, project, and verify that `‖proj‖² + ‖perp‖² = ‖u‖²` — exactly, no rounding.

In [ ]:
def project(u, v):
    return (u.dot_product(v) / v.dot_product(v)) * v

u = vector(QQ, [5, 1])
v = vector(QQ, [3, 4])

p    = project(u, v)
perp = u - p

show('u =', u, '   v =', v)
show('proj_v u =', p)
show('u − proj =', perp)
show('perp · v =', perp.dot_product(v), '  (should be 0)')

show('|proj|² =', p.norm()^2)
show('|perp|² =', perp.norm()^2)
show('|u|²    =', u.norm()^2)
print('Pythagoras holds?', bool(p.norm()^2 + perp.norm()^2 == u.norm()^2))

## 6. Cauchy–Schwarz — symbolic proof in ℝ²

The Cauchy–Schwarz inequality `(u · v)² ≤ ‖u‖² · ‖v‖²` is famously proved by considering the discriminant of `‖u + t v‖²` as a polynomial in t. Sage lets us watch it happen.

Even simpler: in ℝ² the difference `‖u‖²·‖v‖² − (u·v)²` is a *sum of squares*. Sage can show this:

In [ ]:
var('u1 u2 v1 v2')
u_sym = vector(SR, [u1, u2])
v_sym = vector(SR, [v1, v2])

diff = u_sym.norm()^2 * v_sym.norm()^2 - u_sym.dot_product(v_sym)^2
show('|u|² |v|² − (u·v)² =', diff.expand())
show('factored =', factor(diff.expand()))

The difference factors as `(u₁ v₂ − u₂ v₁)²` — a square, hence ≥ 0. So `(u·v)² ≤ ‖u‖² ‖v‖²`. Cauchy–Schwarz, proved in three lines.

Notice also: equality holds iff `u₁ v₂ = u₂ v₁`, which is exactly the condition for u and v to be scalar multiples of each other.

## 7. Parametric line, plotted with arrows

In [ ]:
A = vector([1, 2])
B = vector([4, 8])
v = B - A

# Plot the segment from A to B and the arrow representing v
ts = srange(-0.4, 1.5, 0.05)
pts = [tuple(A + t * v) for t in ts]

p = (
    line(pts, color='blue', alpha=0.5, legend_label='L(t) = A + t·v')
    + point(tuple(A), color='green', size=80, legend_label='A = L(0)')
    + point(tuple(B), color='red',   size=80, legend_label='B = L(1)')
    + point(tuple((A + B) / 2), color='black', size=60, legend_label='midpoint = L(1/2)')
    + arrow2d(tuple(A), tuple(B), color='gray', width=1, linestyle='dashed')
)
p.show(aspect_ratio=1, gridlines=True, frame=True, xmin=-2, xmax=7, ymin=-2, ymax=12)

## 8. Standard unit vectors and the e_i family

Sage knows the standard basis. Any vector decomposes as `v = ∑ vᵢ eᵢ`.

In [ ]:
n = 4
I = identity_matrix(QQ, n)
es = [I.column(i) for i in range(n)]

for i, e in enumerate(es, start=1):
    show(f'e_{i} =', e)

v = vector(QQ, [3, -1, 5, 2])
reconstruction = sum(v[i] * es[i] for i in range(n))
show('v =', v)
show('Σ vᵢ eᵢ =', reconstruction)
print('Match?', bool(v == reconstruction))

Each unit vector has norm 1 and they're mutually perpendicular — the foundation of orthogonality (Chapter 7).

In [ ]:
for i in range(n):
    print(f'|e_{i+1}| = {es[i].norm()}')
for i in range(n):
    for j in range(i+1, n):
        print(f'e_{i+1} · e_{j+1} = {es[i].dot_product(es[j])}')

---

## 9. Exercises

Try each problem before peeking at the Solutions section below.

### Exercise 1 — Exact magnitudes
Compute `‖v‖` for `v = (1, 2, 2, 4)` and for `v = (1, 1, 1, 1, 1, 1, 1, 1, 1)`, both over `QQ`. Express each answer exactly.
Then normalize the second vector and verify the result has norm 1.

In [ ]:
# Your code here


### Exercise 2 — Exact angles
Let `u = (1, 0, 0)` and `v = (1, 1, 1)`. Compute `cos θ` exactly and `θ` in degrees (numerically).

In [ ]:
# Your code here


### Exercise 3 — Find a perpendicular vector
Given `u = (3, 4)` in ℝ², find a vector `v` with `‖v‖ = 1` and `u · v = 0`. (Hint: in ℝ², swap and negate one component, then normalize.) Verify both conditions symbolically.

In [ ]:
# Your code here


### Exercise 4 — Project and decompose
Let `u = (3, 4)` and `v = (1, 1)` (both over QQ). Compute `proj_v u`, the perpendicular remainder, and verify Pythagoras `‖proj‖² + ‖perp‖² = ‖u‖²` exactly.

In [ ]:
# Your code here


### Exercise 5 — Cauchy–Schwarz with equality
Show numerically that `|u · v| < ‖u‖ · ‖v‖` for `u = (1, 2)`, `v = (1, 1)`. Then choose `w = 3u` (a scalar multiple of u) and verify that `|u · w| = ‖u‖ · ‖w‖` (equality case).

In [ ]:
# Your code here


### Exercise 6 — Parametric line through two points
Write a function `line_through(A, B)` that returns the parametric form `t -> A + t·(B - A)`. Test it: confirm `line_through(A, B)(0) == A` and `line_through(A, B)(1) == B` for `A = (2, -1)`, `B = (4, 5)`.

In [ ]:
# Your code here


### Exercise 7 — Distance from a point to a line
Find the perpendicular distance from `P = (7, 0)` to the line through origin in direction `v = (2, 1)`. Express the answer exactly (it should be `7/√5`).

In [ ]:
# Your code here


---

## 10. Solutions

*Try the exercises before reading these.*

### Solution 1

In [ ]:
v1 = vector(QQ, [1, 2, 2, 4])
show('|v1| =', v1.norm())

v2 = vector(QQ, [1]*9)
show('|v2| =', v2.norm())

v2_hat = v2 / v2.norm()
show('v2_hat =', v2_hat)
show('|v2_hat| =', v2_hat.norm())

### Solution 2

In [ ]:
u = vector(QQ, [1, 0, 0])
v = vector(QQ, [1, 1, 1])
cos_t = u.dot_product(v) / (u.norm() * v.norm())
show('cos θ =', cos_t)
show('θ (rad) =', arccos(cos_t))
show('θ (deg) ≈', n(arccos(cos_t) * 180 / pi))

### Solution 3

In [ ]:
u = vector(QQ, [3, 4])
perp_raw = vector(QQ, [-u[1], u[0]])     # swap and negate
v = perp_raw / perp_raw.norm()
show('v =', v)
show('|v| =', v.norm())
show('u · v =', u.dot_product(v))

### Solution 4

In [ ]:
u = vector(QQ, [3, 4])
v = vector(QQ, [1, 1])
p = (u.dot_product(v) / v.dot_product(v)) * v
perp = u - p
show('proj_v u =', p)
show('u − proj =', perp)
show('perp · v =', perp.dot_product(v))
show('|proj|² + |perp|² =', p.norm()^2 + perp.norm()^2)
show('|u|²              =', u.norm()^2)
print('Pythagoras holds?', bool(p.norm()^2 + perp.norm()^2 == u.norm()^2))

### Solution 5

In [ ]:
u = vector(QQ, [1, 2])
v = vector(QQ, [1, 1])
show('|u·v| =', abs(u.dot_product(v)))
show('|u| · |v| =', u.norm() * v.norm())
print('Strict <?', bool(abs(u.dot_product(v)) < u.norm() * v.norm()))

w = 3*u
show('|u·w| =', abs(u.dot_product(w)))
show('|u| · |w| =', u.norm() * w.norm())
print('Equality?', bool(abs(u.dot_product(w)) == u.norm() * w.norm()))

### Solution 6

In [ ]:
def line_through(A, B):
    A = vector(A); B = vector(B)
    return lambda t: A + t * (B - A)

L = line_through((2, -1), (4, 5))
show('L(0)   =', L(0))
show('L(1)   =', L(1))
show('L(1/2) =', L(1/2))

### Solution 7

In [ ]:
P = vector(QQ, [7, 0])
v = vector(QQ, [2, 1])
p = (P.dot_product(v) / v.dot_product(v)) * v
perp = P - p
show('distance =', perp.norm())
show('         =', perp.norm().simplify_full())

---

## Where to next

- **Chapter 3 (Sage):** `A.rref()`, `A.solve_right(b)` — but the dot-product bilinearity from §2 is exactly what makes matrix-vector multiplication tick.
- **Chapter 7 (Sage):** `A.gram_schmidt()`, `A.QR()`, `A.solve_least_squares(...)` — every one of these uses the projection formula from §4 of *this* notebook, generalized to subspaces.
- **Chapter 9 (Sage):** `A.eigenvectors_right()` — eigenvectors are typically *normalized* (using §1 norms) before being reported.